# 02 : Modélisation - Rétention des professionnels de santé

**Entrée** : table `hcp_features` (créée par le notebook 01)
**Unité** : 1 ligne = 1 professionnel de santé | ~5 700 HCP
**Cible** : `retenu` (1 = encore payé en 2023, 0 = non) | taux de base **73,3 %**

---

## Contexte

Un laboratoire a intérêt à savoir **quels professionnels de santé resteront engagés** d'une année sur l'autre : c'est la question de la **rétention** (l'équivalent du churn en analytics client). On prédit la rétention à partir du **profil d'engagement 2022**, on l'explique, on segmente, et on vérifie que le modèle est **équitable** entre spécialités.

## Research Questions

1. **Prédiction** : à partir du profil d'engagement 2022 d'un HCP, peut-on prédire s'il sera encore payé en 2023 ?
2. **Explication** : quels facteurs pilotent la rétention ?
3. **Segmentation** : peut-on regrouper les HCP en profils d'engagement lisibles pour le métier ?
4. **Équité** : le modèle est-il aussi fiable selon la spécialité du professionnel ?

## Variables du modèle

| Type | Variables |
| --- | --- |
| Numériques | `n_payments`, `total_amount`, `mean_amount`, `n_manufacturers`, `n_natures`, part de chaque nature (repas, voyage, conseil, orateur, formation) |
| Catégorielle | `specialty` |
| Cible | `retenu` (0/1) |

## 0. Setup et chargement

On importe les librairies et on charge la table `hcp_features` depuis la base SQLite construite au notebook 01.

In [1]:
import sqlite3, pathlib
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM_FEATURES = ["n_payments", "total_amount", "mean_amount", "median_amount", "n_manufacturers", "n_natures",
                "share_food", "share_travel", "share_consulting", "share_speaker", "share_education"]
CAT_FEATURES = ["specialty"]
TARGET = "retenu"

ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd().parent / "data").exists() else pathlib.Path.cwd()
DB_PATH = ROOT / "data" / "openpayments.sqlite"
with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("SELECT * FROM hcp_features", con)
print("Profils charges:", len(df))

Profils charges: 5707


## 1. Modèle de rétention et évaluation

On compare un **RandomForest** (avec standardisation des variables numériques et encodage one-hot de la spécialité) à une **baseline naïve** (prédire toujours la classe majoritaire). On mesure ROC-AUC et PR-AUC sur un jeu de test stratifié (25 %). La baseline sert de garde-fou : un bon AUC ne vaut que comparé à elle.

In [2]:
def build_pipeline():
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_FEATURES),
    ])
    clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                 class_weight="balanced", random_state=0, n_jobs=-1)
    return Pipeline([("pre", pre), ("clf", clf)])

X = df[NUM_FEATURES + CAT_FEATURES]
y = df[TARGET].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

base = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
pipe = build_pipeline().fit(X_tr, y_tr)
proba = pipe.predict_proba(X_te)[:, 1]

print("Taux de retention   :", round(y.mean(), 3))
print("ROC-AUC baseline    :", round(roc_auc_score(y_te, base.predict_proba(X_te)[:, 1]), 3))
print("ROC-AUC RandomForest:", round(roc_auc_score(y_te, proba), 3))
print("PR-AUC  RandomForest:", round(average_precision_score(y_te, proba), 3))
print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))

Taux de retention   : 0.733
ROC-AUC baseline    : 0.5
ROC-AUC RandomForest: 0.805
PR-AUC  RandomForest: 0.92
              precision    recall  f1-score   support

           0      0.483     0.790     0.600       381
           1      0.900     0.692     0.783      1046

    accuracy                          0.718      1427
   macro avg      0.692     0.741     0.691      1427
weighted avg      0.789     0.718     0.734      1427



> **Observations.** ROC-AUC **0,805** contre **0,500** pour la baseline, PR-AUC 0,920 : le modèle sépare nettement les HCP retenus des non-retenus. La classe minoritaire (non-retenus) reste plus difficile (rappel plus faible), attendu vu le déséquilibre 73/27.

## 2. Explicabilité : importances et SHAP

Deux lectures complémentaires : les **importances** du RandomForest (contribution globale de chaque variable) et **SHAP** (contribution moyenne, plus robuste et locale). Si les deux concordent, l'interprétation est fiable.

In [3]:
names = pipe.named_steps["pre"].get_feature_names_out()
imp = pd.Series(pipe.named_steps["clf"].feature_importances_, index=names).sort_values(ascending=False)
print("Importances (top 10):")
print(imp.head(10).round(3).to_string())

try:
    import shap
    Xs = df[NUM_FEATURES + CAT_FEATURES].sample(min(500, len(df)), random_state=0)
    Xt = pipe.named_steps["pre"].transform(Xs)
    vals = shap.TreeExplainer(pipe.named_steps["clf"]).shap_values(Xt)
    arr = vals[1] if isinstance(vals, list) else np.asarray(vals)
    if arr.ndim == 3:
        arr = arr[:, :, 1]
    top = pd.Series(np.abs(arr).mean(axis=0), index=names).sort_values(ascending=False).head(10)
    print("\nSHAP (contributions moyennes):")
    print(top.round(4).to_string())
except ImportError:
    print("(SHAP non installe : pip install shap)")

Importances (top 10):
num__n_payments                                                              0.261
num__n_manufacturers                                                         0.233
num__total_amount                                                            0.207
num__median_amount                                                           0.112
num__mean_amount                                                             0.109
num__share_food                                                              0.017
num__n_natures                                                               0.015
num__share_education                                                         0.011
cat__specialty_Physician Assistants & Advanced Practice Nursing Providers    0.009
cat__specialty_Allopathic & Osteopathic Physicians                           0.009


F:\Spyder\envs\ptr2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



SHAP (contributions moyennes):
num__n_payments                                                              0.1123
num__n_manufacturers                                                         0.1009
num__total_amount                                                            0.0628
num__median_amount                                                           0.0223
num__mean_amount                                                             0.0186
cat__specialty_Allopathic & Osteopathic Physicians                           0.0080
cat__specialty_Physician Assistants & Advanced Practice Nursing Providers    0.0065
num__n_natures                                                               0.0062
num__share_food                                                              0.0057
num__share_education                                                         0.0034


> **Observations.** Importances et SHAP **concordent** : la rétention est pilotée par l'**intensité d'engagement** (`n_payments`, `n_manufacturers`, `total_amount`), pas par la spécialité. Le `median_amount`, ajouté après correction des décisions du notebook 01, apporte un signal complémentaire robuste aux valeurs extrêmes. Autrement dit, plus un HCP est sollicité par de nombreux labos et souvent, plus il reste engagé l'année suivante.

## 3. Segmentation des profils d'engagement (k-means)

Au-delà de la prédiction, on regroupe les HCP en **profils types** par k-means (4 clusters) sur les variables numériques standardisées. On lit chaque segment via ses moyennes (montant, nombre de paiements, nombre de labos) et son taux de rétention.

In [4]:
# Transformation log1p sur les variables tres asymetriques avant clustering
# (decision EDA notebook 01) : evite qu un HCP extreme domine la distance
SKEWED = ["n_payments", "total_amount", "mean_amount", "median_amount", "n_manufacturers", "n_natures"]
Xseg_raw = df[NUM_FEATURES].fillna(0).copy()
Xseg_raw[SKEWED] = np.log1p(Xseg_raw[SKEWED])
Xseg = StandardScaler().fit_transform(Xseg_raw)
d = df.copy()
d["segment"] = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(Xseg)
seg = d.groupby("segment").agg(
    n_hcp=("segment", "size"),
    total_amount_moyen=("total_amount", "mean"),
    n_payments_moyen=("n_payments", "mean"),
    n_labos_moyen=("n_manufacturers", "mean"),
    taux_retention=(TARGET, "mean"),
).round(2)
seg

,n_hcp,total_amount_moyen,n_payments_moyen,n_labos_moyen,taux_retention
segment,,,,,
0,3384,76.83,2.69,1.67,0.62
1,1630,756.37,39.35,9.82,0.96
2,512,1891.86,10.73,2.86,0.73
3,181,27597.23,59.83,6.82,0.87


> **Observations.** Quatre profils se dégagent, recalculés après application du log1p sur les variables très asymétriques (décision EDA notebook 01) : un large groupe à **engagement minimal** (3 384 HCP, ~2,7 paiements, 1,7 labo, **62 %** de rétention), un **cœur fidèle** très sollicité (1 630 HCP, ~39 paiements, ~10 labos, **96 %**), un segment à **valeur ponctuelle** (512 HCP, montant moyen ~1 892 $, engagement modéré, 73 %), et un groupe **haute valeur** restreint (181 HCP, montant moyen ~27 597 $, 87 %). La fréquence et la diversité de laboratoires distinguent mieux les segments que le seul montant : un ciblage actionnable pour prioriser les relances commerciales.

## 4. Équité (fairness) : performance par spécialité

Un modèle peut être globalement bon mais **injuste** selon les sous-groupes. On évalue l'AUC **par spécialité** avec des probabilités hors-échantillon (validation croisée 5 folds), pour vérifier que le modèle est fiable partout, pas seulement sur les groupes majoritaires.

In [5]:
proba_cv = cross_val_predict(build_pipeline(), X, y, cv=5, method="predict_proba", n_jobs=-1)[:, 1]
d2 = df.copy(); d2["proba"] = proba_cv
print(f"  {'specialite':45s}  {'n':>5s}  {'retention':>9s}  {'AUC':>6s}")
for sp in d2["specialty"].value_counts().head(6).index:
    m = d2["specialty"] == sp
    if m.sum() >= 50 and d2.loc[m, TARGET].nunique() > 1:
        auc = roc_auc_score(d2.loc[m, TARGET], d2.loc[m, "proba"])
        print(f"  {str(sp)[:45]:45s}  {m.sum():5d}  {d2.loc[m, TARGET].mean():9.2f}  {auc:6.3f}")

  specialite                                         n  retention     AUC
  Allopathic & Osteopathic Physicians             2852       0.72   0.807
  Physician Assistants & Advanced Practice Nurs   2272       0.77   0.804
  Dental Providers                                 358       0.61   0.639
  Eye and Vision Services Providers                158       0.85   0.808
  Podiatric Medicine & Surgery Service Provider     57       0.82   0.898


> **Observations.** Le modèle est solide et homogène sur les **gros groupes** (médecins ~0,81, PA/IPA ~0,80) mais **nettement plus faible pour les dentistes (~0,64)**. Conclusion d'IA responsable : il **ne doit pas être appliqué uniformément** ; pour les groupes sous-performants, prévoir un indicateur de faible confiance ou un modèle dédié. (Voir `responsible_ai/model_card.md`.)